**Created by Tahim Bhuiya**

**Part 6**

**09/20/2026**

Application of Keras to build, compile, and train a neural network as a three-class classifier for MNIST dataset (0 vs. 3 vs. 8):

a) Use mnist function in keras.datasets to load MNIST dataset and split it into training and testing sets. Then, randomly select 20% of the training images along with their corresponding labels to be the validation data.

In [1]:
import numpy as np
from keras.datasets import mnist

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Select only classes 0, 3, 8
x_train_038 = x_train[np.logical_or.reduce([y_train==0, y_train==3, y_train==8])]
y_train_038 = y_train[np.logical_or.reduce([y_train==0, y_train==3, y_train==8])]

x_test_038 = x_test[np.logical_or.reduce([y_test==0, y_test==3, y_test==8])]
y_test_038 = y_test[np.logical_or.reduce([y_test==0, y_test==3, y_test==8])]

# Create validation split (20% of training)
num_samples = x_train_038.shape[0]
indices = np.random.permutation(num_samples)

val_size = int(0.2 * num_samples)
val_idx = indices[:val_size]
train_idx = indices[val_size:]

x_val_038, y_val_038 = x_train_038[val_idx], y_train_038[val_idx]
x_train_038, y_train_038 = x_train_038[train_idx], y_train_038[train_idx]

# Print shapes to check
print("Training set:", x_train_038.shape)
print("Validation set:", x_val_038.shape)


Training set: (14324, 28, 28)
Validation set: (3581, 28, 28)


b) Feature extraction: average the pixel values in the quadrants in each image to generate a feature vector of 4 values for each image.

In [2]:
def feat_extract(images):
    width = images.shape[1]
    height = images.shape[2]

    features = np.zeros((images.shape[0], 4))

    # Quadrant 0 (top left)
    features_temp = np.sum(images[:, 0:int(width/2), 0:int(height/2)], axis=2)
    features[:, 0] = np.sum(features_temp, axis=1) / (width * height / 4)

    # Quadrant 1 (top right)
    features_temp = np.sum(images[:, 0:int(width/2), int(height/2):], axis=2)
    features[:, 1] = np.sum(features_temp, axis=1) / (width * height / 4)

    # Quadrant 2 (bottom left)
    features_temp = np.sum(images[:, int(width/2):, 0:int(height/2)], axis=2)
    features[:, 2] = np.sum(features_temp, axis=1) / (width * height / 4)

    # Quadrant 3 (bottom right)
    features_temp = np.sum(images[:, int(width/2):, int(height/2):], axis=2)
    features[:, 3] = np.sum(features_temp, axis=1) / (width * height / 4)

    return features

In [3]:
# Apply feature extraction to all datasets
x_train_feat = feat_extract(x_train_038)
x_val_feat   = feat_extract(x_val_038)
x_test_feat  = feat_extract(x_test_038)

# Check shapes
print("x_train features shape:", x_train_feat.shape)
print("x_val features shape:", x_val_feat.shape)
print("x_test features shape:", x_test_feat.shape)

x_train features shape: (14324, 4)
x_val features shape: (3581, 4)
x_test features shape: (2964, 4)


c) Convert the label vectors for all the sets to binary class matrices using to_categorical() Keras function.


In [4]:
from keras.utils import to_categorical

# Define classes and map labels to 0,1,2
classes = [0, 3, 8]
label_map = {0:0, 3:1, 8:2}

# Map filtered labels to 0,1,2
y_train_038_mapped = np.array([label_map[y] for y in y_train_038])
y_val_038_mapped   = np.array([label_map[y] for y in y_val_038]) # Mapping was needed to produce a proper vector
y_test_038_mapped  = np.array([label_map[y] for y in y_test_038])

# Convert
y_train_038_c = to_categorical(y_train_038_mapped, len(classes))
y_val_038_c   = to_categorical(y_val_038_mapped, len(classes))
y_test_038_c  = to_categorical(y_test_038_mapped, len(classes))

# Checking shapes
print("y_train_038_c shape:", y_train_038_c.shape)
print("y_val_038_c shape:", y_val_038_c.shape)
print("y_test_038_c shape:", y_test_038_c.shape)

y_train_038_c shape: (14324, 3)
y_val_038_c shape: (3581, 3)
y_test_038_c shape: (2964, 3)


d) Build, compile, train, and then evaluate:

i. Build a neural network with 1 layer that contains 16 nodes using the Keras library.

In [5]:
from keras.models import Sequential
from keras.layers import Dense

# Build the model
model = Sequential()

# Hidden layer with 16 nodes
model.add(Dense(input_dim=4, units=16, activation='tanh'))

# Output layer with 3 nodes and softmax activation
model.add(Dense(units=3, activation='softmax'))

# Model summary
model.summary()

C:\Users\tahim\tf_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 16)                  │              80 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 131 (524.00 B)

 Trainable params: 131 (524.00 B)

 Non-trainable params: 0 (0.00 B)

ii. Compile the network. Make sure to select the correct loss function for this classification problem. Use stochastic gradient descent learning (SGD, learning rate of 0.0001). Explain your selection of the loss function.

In [6]:
from keras.optimizers import SGD

# Define the optimizer
optimizer = SGD(learning_rate=0.0001)

# Compile the model
model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

Explain your selection of the loss function:

I selected categorical crossentropy as the loss function because this is a multiclass classification problem with three classes (0, 3, 8). Categorical crossentropy measures how far the predicted probabilities are from the true labels as well.

iii. Train the network for 30 epochs and a batch size of 16.

In [7]:
# Train the model
history = model.fit(
    x_train_feat,
    y_train_038_c,
    epochs=30,
    batch_size=16,
    verbose=1,
    validation_data=(x_val_feat, y_val_038_c)
)

Epoch 1/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.3972 - loss: 1.1367 - val_accuracy: 0.4731 - val_loss: 1.0626
Epoch 2/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.4695 - loss: 1.0542 - val_accuracy: 0.4873 - val_loss: 1.0297
Epoch 3/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.4881 - loss: 1.0250 - val_accuracy: 0.5001 - val_loss: 1.0094
Epoch 4/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5031 - loss: 1.0076 - val_accuracy: 0.5099 - val_loss: 0.9951
Epoch 5/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5094 - loss: 0.9960 - val_accuracy: 0.5135 - val_loss: 0.9849
Epoch 6/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5144 - loss: 0.9873 - val_accuracy: 0.5186 - val_loss: 0.9764
Epoch 7/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5188 - loss: 0.9803 - val_accuracy: 0.5242 - val_loss: 0.9703
Epoch 8/30
896/896 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5208 - loss: 0.9741 - val_accuracy: 0.

iv. Plot the training loss (i.e., the learning curve) for all the epochs.

In [ ]:
import matplotlib.pyplot as plt

# Get loss values from history
train_loss = history.history['loss']
epochs = range(1, len(train_loss) + 1)

# Plot the training loss
plt.figure(figsize=(8,5))
plt.plot(epochs, train_loss, 'b-', label='Training Loss')



